In [ ]:
# Extract, filter and compile datasets

import pandas as pd

def exfilter(date,i,n):
    """Extract and filter the dataset from an individual file as a pandas DataFrame."""
    # Construct the file suffix, accounting for single-digit and double-digit part numbers
    if n > 9:
        z = ""
    else:
        z = "0"
    # Read DataFrame
    df = pd.read_stata(f'output_{str(date[i])}_part-000{z+str(n)}.dta')
    df_length=len(df)
    # Filter for Engineering
    fdf=df[df["title"].str.contains(r"engineer", case=False, na=False)]
    return fdf, df_length

# Dates corresponding to the selected dataset snapshots
date = (
    '2022_10_2', '2023_1_1', '2023_4_2', '2023_7_5',
    '2023_10_1', '2024_1_7', '2024_4_7', '2024_7_7',
    '2024_10_6', '2025_1_5', '2025_4_6', '2025_7_6'
)

relevance = []
fdf_length = []

for i in range(len(date)):
    df_length = 0
    fdf = []
    # Define the range of dataset parts for each snapshot
    if i == 0:
        x=0
        y=7
    elif i == 1:
        x=2
        y=10
    elif i in (6,7,8,10):
        x=0
        y=11
    elif i == 11:
        x=0
        y=12
    else:
        x=0
        y=10

    # Extract and filter each dataset part
    for n in range(x,y):
        fdf_n, df_length_n = exfilter(date, i, n)
        fdf.append(fdf_n)
        df_length += df_length_n

    # Combine filtered dataset parts into a single DataFrame
    fdf = pd.concat(fdf, ignore_index=True)
    fdf_length.append(len(fdf))
    relevance.append(round(len(fdf)/df_length*100,3))

    print(f"Filtered dataset for {date[i]}: {len(fdf)} of {df_length} ({relevance[i]}%)")

    # Save the filtered dataset for the current snapshot
    fdf.to_csv(f"efdf_{date[i]}.csv", index=False)

print("\nFilter completed.")

Filtered Dataset for 2022_10_2: 82838 of 1083525 (7.645%)
Filtered Dataset for 2023_1_1: 77092 of 894938 (8.614%)
Filtered Dataset for 2023_4_2: 85585 of 1059672 (8.077%)
Filtered Dataset for 2023_7_5: 83733 of 1064712 (7.864%)
Filtered Dataset for 2023_10_1: 89097 of 1092172 (8.158%)
Filtered Dataset for 2024_1_7: 62668 of 797452 (7.859%)
Filtered Dataset for 2024_4_7: 68784 of 863310 (7.967%)
Filtered Dataset for 2024_7_7: 66420 of 861690 (7.708%)
Filtered Dataset for 2024_10_6: 66225 of 879442 (7.53%)
Filtered Dataset for 2025_1_5: 53435 of 743726 (7.185%)
Filtered Dataset for 2025_4_6: 60626 of 855353 (7.088%)
Filtered Dataset for 2025_7_6: 62141 of 883207 (7.036%)


In [ ]:
# Extract a 12,000-sample dataset (combined)

import pandas as pd

# Dates corresponding to the selected dataset snapshots
date = (
    '2022_10_2', '2023_1_1', '2023_4_2', '2023_7_5',
    '2023_10_1', '2024_1_7', '2024_4_7', '2024_7_7',
    '2024_10_6', '2025_1_5', '2025_4_6', '2025_7_6'
)

sample_df=[]

for i in range(0,len(date)):
    df = pd.read_csv(f"efdf_{date[i]}.csv")
    # Randomly sample 1000 records from each snapshot date
    sample = df.sample(n=1000, random_state=42)
    # Record the snapshot date/quarter associated with each record
    sample["quarter"] = date[i]
    sample_df.append(sample)

    print(f"Sampling {date[i]}")

# Combine samples from all snapshot dates into a single dataset
df_all = pd.concat(sample_df, ignore_index=True)
# Save the combined 12000-sample
df_all.to_csv("efdf_all.csv", index=False)

print("\nSample created!")

Sampling 2022_10_2
Sampling 2023_1_1
Sampling 2023_4_2
Sampling 2023_7_5
Sampling 2023_10_1
Sampling 2024_1_7
Sampling 2024_4_7
Sampling 2024_7_7
Sampling 2024_10_6
Sampling 2025_1_5
Sampling 2025_4_6
Sampling 2025_7_6
Sample created!


In [1]:
# VSRS

import pandas as pd
import re

def vsrs(description):
    """Extract visa/sponsorship/right-to-work statements (VSRS)."""

    keywords = [
        "visa",
        "sponsor",
        "right to live and work",
        "right to work",
        "eligible to work",
        "eligibility to work",
        "work eligibility",
        "authorised to work",
        "authorisation to work",
        "authorized to work",
        "authorization to work",
        "working right",
        "work authorisation",
        "work authorization",
        "permission to work",
        "work permit",
        "working permit",
        "work permission",
        "working permission",
    ]

    statements = []

    # Split the description into sentences and bullet-point statements
    sentences = re.split(r'(?<=[.!?])\s+|\s*[-*+·•]\s*', description)

    # Find and store statements containing at least one VSRS keyword
    for sentence in sentences:
        if any(keyword in sentence.lower() for keyword in keywords):
            statements.append(sentence.strip())

    # Return combined statement(s) and remove duplicates (preserving original order)
    return " ".join(dict.fromkeys(statements)) or "Not found"

# Extract VSRS statements to new column and create a separate dataset containing only jobs with VSRS
df = pd.read_csv("efdf_all.csv")
df["vsrs"] = df["description"].apply(vsrs)
vsrs_df = df[df["vsrs"] != "Not found"].copy()

# Determining the number of jobs with VSRS
vsrs_jobs = (df["vsrs"] != "Not found").sum()
print(f"Jobs with VSRS: {vsrs_jobs} of {len(df)}")

# Save the complete dataset with the VSRS column
df.to_csv("efdf_all_vsrs_applied.csv", index=False)
# Save only the jobs containing VSRS statements
vsrs_df.to_csv("efdf_all_vsrs_filtered.csv", index=False)

Jobs with VSRS: 1086 of 12000
